# Credit Risk Decision Engine & Business Summary

## Objective

This notebook consolidates the outputs generated throughout the credit risk modeling pipeline into business-focused summaries.

The notebook combines:

- Probability of Default (PD) model performance
- SHAP explainability results
- Risk segmentation
- Loan approval decisions
- Portfolio optimization
- Executive summary metrics

These outputs demonstrate how machine learning predictions can be translated into practical lending decisions and portfolio management strategies.


## Import Required Libraries

Load the required libraries for reading model outputs and creating executive summary tables.

In [2]:
import pandas as pd
import numpy as np

In [3]:
pd.set_option('display.max_columns',150)
pd.set_option('display.max_rows',50)

## Load Final Model Outputs

Load the outputs generated from previous notebooks, including model performance, explainability results, risk segmentation summaries, lending decisions, and portfolio optimization metrics.

In [4]:
pd_results = pd.read_csv("../data/pd_model_results.csv")
shap_importance = pd.read_csv("../data/shap_feature_importance.csv")
risk_band_summary = pd.read_csv("../data/risk_band_summary.csv")
decision_summary = pd.read_csv("../data/decision_summary.csv")
portfolio_comparison = pd.read_csv("../data/portfolio_comparison.csv")


## Review Probability of Default Model Performance

Compare the performance of all trained Probability of Default (PD) models using multiple evaluation metrics.

Key evaluation metrics include:

- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC

The highest-performing model will be selected for deployment.

In [5]:
pd_results.sort_values('roc_auc', ascending=False)

,model,accuracy,precision,recall,f1_score,roc_auc
0,LightGBM,0.69605,0.391840,0.674266,0.495644,0.762462
1,Random Forest Classifier,0.65680,0.357193,0.687133,0.470043,0.732993
2,Logistic Regression,0.60865,0.299563,0.573025,0.393444,0.635850


## Review Top SHAP Risk Drivers

SHAP (SHapley Additive exPlanations) values identify the features that contribute most to model predictions.

The top-ranked features provide insight into which borrower characteristics most strongly influence default risk.

In [6]:
top_shap_features = shap_importance.head(20)
top_shap_features

,feature,mean_abs_shap
0,sub_grade_num,0.310458
1,issue_year,0.232436
2,total_rec_late_fee,0.182664
3,term_months,0.182460
4,grade_num,0.079077
5,acc_open_past_24mths,0.076200
6,avg_cur_bal,0.068816
7,financial_stability_Score,0.068480
8,creditworthiness_score,0.064170
9,num_actv_rev_tl,0.058122


## Risk Brand Summary

In [7]:
risk_band_summary

,risk_band,borrower_count,default_count,default_rate,min_pd_score,max_pd_score,avg_pd_score
0,Very Low,8000,389,0.048625,0.025880,0.230880,0.155090
1,Low,8000,860,0.107500,0.230932,0.358714,0.294819
2,Medium,8000,1473,0.184125,0.358716,0.486674,0.422031
3,High,8000,2111,0.263875,0.486684,0.641309,0.560484
4,Very High,8000,4027,0.503375,0.641349,0.967616,0.757968


In [8]:
## Loan Decision Summary

In [9]:
decision_summary

,loan_decision,borrower_count,actual_default_rate,total_lgd,avg_lgd,total_ead,avg_ead,total_expected_loss,avg_expected_loss,total_risk_adjusted_return,avg_risk_adjusted_return,total_expected_interest_income,avg_expected_interest_income
0,Approve,3799,0.039221,2069.072537,0.544636,52302225,13767.366412,3.520841e+06,926.780870,8.843959e+05,232.797023,4.405236e+06,1159.577894
1,Manual Review,8000,0.184125,4790.797142,0.598850,108561100,13570.137500,2.753516e+07,3441.894746,-1.297154e+07,-1621.442993,1.456361e+07,1820.451753
2,Reject,28201,0.256658,17057.130034,0.604841,419678975,14881.705436,1.354806e+08,4804.107866,-7.433020e+07,-2635.729333,6.115044e+07,2168.378533


## Portfolio Optimization Summary

In [10]:
portfolio_comparison

,portfolio,borrower_count,actual_default_rate,avg_pd,avg_lgd,total_ead,avg_ead,total_expected_loss,avg_expected_loss,total_expected_interest_income,avg_expected_interest_income,total_risk_adjusted_return,avg_risk_adjusted_return
0,Eligible Portfolio,11799,0.137469,0.326267,0.581394,160863325,13633.640563,3.105600e+07,2632.087337,1.896885e+07,1607.665941,-1.208715e+07,-1024.421396
1,Optimized Portfolio,5000,0.076800,0.192400,0.561357,56814975,11362.995000,4.634156e+06,926.831193,5.074427e+06,1014.885305,4.402706e+05,88.054113


## Final Executive Summary Table

In [11]:
best_model = pd_results.sort_values("roc_auc", ascending=False).iloc[0]

final_summary = pd.DataFrame({
    "metric": [
        "Best PD Model",
        "Best Model ROC-AUC",
        "Best Model Recall",
        "Best Model Precision",
        "Number of Risk Bands",
        "Top SHAP Feature",
        "Portfolio Comparison Available"
    ],
    "value": [
        best_model["model"],
        round(best_model["roc_auc"], 4),
        round(best_model["recall"], 4),
        round(best_model["precision"], 4),
        risk_band_summary.shape[0],
        top_shap_features.iloc[0]["feature"],
        "Yes"
    ]
})

final_summary

,metric,value
0,Best PD Model,LightGBM
1,Best Model ROC-AUC,0.7625
2,Best Model Recall,0.6743
3,Best Model Precision,0.3918
4,Number of Risk Bands,5
5,Top SHAP Feature,sub_grade_num
6,Portfolio Comparison Available,Yes


## Save Final Outputs 

In [12]:
final_summary.to_csv("../data/final_project_summary.csv", index=False)

top_shap_features.to_csv("../data/top_20_shap_features.csv", index=False)